# Tutorial 13 — Efficient inference and quantisation

[Lecture notes](../lectures/13_efficiency.md) · [Slides](../slides/13_efficiency.pptx) · [Exercise sheet](../exercises/13_efficiency.md)

**40 minutes, CPU, NumPy only.** You will work out what fits on a free GPU, quantise a weight matrix, test whether a steering direction survives quantisation, and check that speculative sampling keeps the target distribution. The memory figures are arithmetic and the matrices are random. Nothing here times or measures a real model.

| Minutes | Step |
|---|---|
| 0–10 | Sections 1–2: weight memory, a fit table, the KV cache |
| 10–20 | Sections 3–4: per-channel quantisation, ideal and actual storage |
| 20–30 | Sections 5–6: a direction under quantisation, speculative sampling |
| 30–40 | Your turn: two functions with tests |

In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)

Python 3.13.1 NumPy 2.1.3 seed 42


## 1. What fits: weight memory

Ideal weight storage is `parameters × bits / 8` bytes. The table uses nominal sizes from the model names, except Llama-3.1-8B, which has 8.03 billion weights. The fit rule is this course's assumption: leave 3 GB free for activations, the KV cache, the runtime and quantisation overhead. A free Colab T4 has 15 GB, so the budget is 12 GB. An 8 GB laptop with 4 GB free for the model has a budget of 4 GB.

Before running: does Llama-3.1-8B fit a T4 at 16 bits? At 8 bits?

In [2]:
models = {"SmolVLM-256M": 0.256, "SmolVLM-500M": 0.5, "Gemma 3 1B": 1.0, "SmolVLM 2.2B": 2.2,
          "Gemma 3 4B": 4.0, "Llama-3.1-8B": 8.03, "Gemma 3 12B": 12.0}
def weights_gb(billions, bits): return billions * bits / 8          # 1e9 weights x bits/8 bytes = GB

T4_BUDGET, LAPTOP_BUDGET = 15 - 3, 4
print(f"{'model':14s}  {'32-bit':>7s} {'16-bit':>7s} {'8-bit':>7s} {'4-bit':>7s}   T4 (12 GB budget)   laptop (4 GB budget)")
fit = {}
for name, n in models.items():
    gb = [weights_gb(n, b) for b in (32, 16, 8, 4)]
    t4 = [b for b, g in zip((32, 16, 8, 4), gb) if g < T4_BUDGET]; cpu = [b for b, g in zip((32, 16, 8, 4), gb) if g < LAPTOP_BUDGET]
    fit[name] = (t4, cpu)
    print(f"{name:14s}  " + " ".join(f"{g:7.2f}" for g in gb) + f"   {str(t4):18s}  {cpu}")
assert np.isclose(weights_gb(8.03, 16), 16.06) and 16 not in fit["Llama-3.1-8B"][0] and 4 in fit["Llama-3.1-8B"][0]

model            32-bit  16-bit   8-bit   4-bit   T4 (12 GB budget)   laptop (4 GB budget)
SmolVLM-256M       1.02    0.51    0.26    0.13   [32, 16, 8, 4]      [32, 16, 8, 4]
SmolVLM-500M       2.00    1.00    0.50    0.25   [32, 16, 8, 4]      [32, 16, 8, 4]
Gemma 3 1B         4.00    2.00    1.00    0.50   [32, 16, 8, 4]      [16, 8, 4]
SmolVLM 2.2B       8.80    4.40    2.20    1.10   [32, 16, 8, 4]      [8, 4]
Gemma 3 4B        16.00    8.00    4.00    2.00   [16, 8, 4]          [4]
Llama-3.1-8B      32.12   16.06    8.03    4.01   [8, 4]              []
Gemma 3 12B       48.00   24.00   12.00    6.00   [4]                 []


Llama-3.1-8B needs 16.06 GB at 16 bits, more than the T4 holds before anything else is loaded. At 8 bits (8.03 GB) and 4 bits (4.01 GB) it is inside the 12 GB budget. On the laptop it fits at no precision under this rule, and the largest model that fits at 16 bits is Gemma 3 1B. The table counts ideal weight storage only. The lecture's working figure for a loaded 4-bit Llama-3.1-8B is about 5.5 GB with overhead.

## 2. The KV cache

During generation each layer stores a key and a value vector for every token already processed, so that the next token does not recompute them. The count is `2 × batch × tokens × layers × KV heads × head dimension`, times the bytes per number.

The first line is a 24-layer model with 8 KV heads of dimension 64. The loop is Llama-3.1-8B: 32 layers, 8 KV heads of dimension 128, 16-bit numbers. Its 32 query heads share those 8 KV heads (grouped-query attention).

Before running: what happens to the cache when the context doubles?

In [3]:
small = 2 * 1 * 4096 * 24 * 8 * 64 * 2
print("24 layers, 8 KV heads x 64, 4,096 tokens:", small, "bytes =", small / 2**20, "MiB")
assert small == 201_326_592
print("Llama-3.1-8B, batch 1, 16-bit cache")
for T in [1024, 4096, 16384, 131072]:
    shared = 2 * 1 * T * 32 * 8 * 128 * 2; unshared = 2 * 1 * T * 32 * 32 * 128 * 2
    print(f"  {T:7d} tokens: {shared / 2**30:6.3f} GiB with 8 KV heads   {unshared / 2**30:6.2f} GiB if all 32 heads kept their own")

24 layers, 8 KV heads x 64, 4,096 tokens: 201326592 bytes = 192.0 MiB
Llama-3.1-8B, batch 1, 16-bit cache
     1024 tokens:  0.125 GiB with 8 KV heads     0.50 GiB if all 32 heads kept their own
     4096 tokens:  0.500 GiB with 8 KV heads     2.00 GiB if all 32 heads kept their own
    16384 tokens:  2.000 GiB with 8 KV heads     8.00 GiB if all 32 heads kept their own
   131072 tokens: 16.000 GiB with 8 KV heads    64.00 GiB if all 32 heads kept their own


The cache grows in proportion to the context: 0.5 GiB at 4,096 tokens and 2 GiB at 16,384. At the model's full 131,072-token context it is 16 GiB, as large as the 16-bit weights. Sharing 8 KV heads among 32 query heads cuts the cache by a factor of four. In common 4-bit set-ups the weights are 4-bit and the cache stays at 16 bits, so quantising the weights does not shrink it.

## 3. Quantise and reconstruct

Symmetric quantisation stores each weight as an integer between −qmax and qmax, times a scale. `W` has 64 inputs and 32 outputs, and output channel 0 is 30 times larger than the rest. Per-tensor quantisation uses one scale for the whole matrix. Per-channel quantisation uses one scale per output column. The output error is measured on random inputs `X`.

Before running: which setting will the large channel hurt most?

In [4]:
W=rng.normal(size=(64,32)); W[:,0]*=30
X=rng.normal(size=(128,64))
def quantize(W,bits,per_channel=True):
    qmax=2**(bits-1)-1
    maxabs=np.max(np.abs(W),axis=0,keepdims=True) if per_channel else np.max(np.abs(W))
    scale=np.where(maxabs==0,1,maxabs/qmax)
    q=np.clip(np.rint(W/scale),-qmax,qmax).astype(np.int8)
    return q,scale,q*scale
quant_mse = {}
for bits in [4,8]:
    for pc in [False,True]:
        q,s,Wq=quantize(W,bits,pc)
        quant_mse[(bits,pc)] = np.mean((W-Wq)**2)
        print(bits,"per_channel",pc,"weight MSE",np.mean((W-Wq)**2),"output MSE",np.mean((X@W-X@Wq)**2))
assert np.isfinite(quantize(np.zeros((2,3)),4)[2]).all()
q4,_,_=quantize(W,4,False)
print("4-bit per-tensor: share of weights outside channel 0 rounded to zero:", round(float((q4[:,1:]==0).mean()),3))

4 per_channel False weight MSE 1.1978280597259148 output MSE 78.49535052190845
4 per_channel True weight MSE 0.23210387377363365 output MSE 15.42621428265472
8 per_channel False weight MSE 0.020517345746461427 output MSE 1.3266943933137862
8 per_channel True weight MSE 0.0006133384432108045 output MSE 0.0359336578001878
4-bit per-tensor: share of weights outside channel 0 rounded to zero: 1.0


With one scale for the whole matrix at 4 bits, channel 0 sets the step and every weight in the other 31 channels rounds to zero. Per-channel scales cut the weight error from 1.20 to 0.23 at 4 bits and from 0.0205 to 0.0006 at 8 bits. The output error keeps the same order: 78.5, 15.4, 1.33 and 0.036. Which is larger here, the gain from per-channel scales at 4 bits or the gain from going to 8 bits per tensor?

## 4. Ideal storage and actual storage

NumPy has no 4-bit type, so the 4-bit integers sit in one byte each. A packed format would put two in a byte. The scales are stored once per channel.

In [5]:
q,s,_=quantize(W,4)
ideal_packed=W.size*4/8+s.size*4 # assume float32 scales
print("ideal packed bytes with float32 scales",ideal_packed)
print("actual NumPy bytes",q.nbytes+s.nbytes)
print("float32 baseline bytes",W.size*4)
assert q.nbytes==W.size

ideal packed bytes with float32 scales 1152.0
actual NumPy bytes 2304
float32 baseline bytes 8192


Packed 4-bit storage would take 1,152 bytes against 8,192 for float32, a factor of 7.1. The 128 bytes of scales are why the factor is not 8. NumPy holds the same integers in 2,304 bytes, because each takes a whole byte and the scales are float64. Bits per weight on paper and bytes in memory are different numbers. Report the one you measured.

## 5. Does a steering direction survive quantisation?

Task 7 teams may find a steering vector on a 4-bit copy of Llama-3.1-8B, because the 16-bit model does not fit a T4. The organisers run the model at full precision. This toy asks the question on a two-layer network: compute the difference-of-means direction between two groups of inputs at full precision, compute it again with quantised weights, and take the cosine between the two.

Two layers of 128 units stand in for a model. The size of the effect in a 32-layer model is unknown until you measure it.

In [6]:
d = 128
W1, W2 = rng.normal(size=(d, d)) / np.sqrt(d), rng.normal(size=(d, d)) / np.sqrt(d)
lang = rng.normal(size=d) / np.sqrt(d)
content = rng.normal(size=(400, d))
X_a, X_b = content - lang, content + lang                    # the same content in two "languages"

def direction(A, B):
    h = lambda x: np.tanh(np.tanh(x @ A) @ B)
    return h(X_b).mean(0) - h(X_a).mean(0)

v_full = direction(W1, W2)
cosines = {}
for bits in [8, 6, 4, 3, 2]:
    v_q = direction(quantize(W1, bits)[2], quantize(W2, bits)[2])
    cosines[bits] = float(v_q @ v_full / np.linalg.norm(v_q) / np.linalg.norm(v_full))
    print(f"{bits} bits: cosine with the full-precision direction {cosines[bits]:.4f}   norm ratio {np.linalg.norm(v_q) / np.linalg.norm(v_full):.3f}")
assert cosines[8] > 0.999 and cosines[2] < cosines[4]

8 bits: cosine with the full-precision direction 1.0000   norm ratio 1.000
6 bits: cosine with the full-precision direction 0.9994   norm ratio 0.997
4 bits: cosine with the full-precision direction 0.9879   norm ratio 1.015
3 bits: cosine with the full-precision direction 0.9354   norm ratio 1.005
2 bits: cosine with the full-precision direction 0.5280   norm ratio 1.049


At 8 bits the direction matches the full-precision one to four decimal places. At 4 bits the cosine is 0.988, at 3 bits 0.935 and at 2 bits 0.528. In this toy a 4-bit direction is close to the full-precision one. A real model has 32 layers through which the error can grow, so run the comparison on a model you can load at both precisions, and score compliance and relevance with the 4-bit vector applied to the 16-bit model.

## 6. Speculative sampling keeps the target distribution

A small draft model proposes a token from its distribution `q`. The large target model has distribution `p`. Accept the token with probability `min(1, p/q)`. On rejection, sample from the leftover `max(p − q, 0)`, renormalised. The claim is that the token you end up with is distributed exactly as `p`.

Before running: the acceptance rate equals the overlap of the two distributions, the sum of `min(p, q)`. Compute it by hand.

In [7]:
p = np.array([0.50, 0.25, 0.15, 0.07, 0.03])        # target model
q_draft = np.array([0.30, 0.40, 0.10, 0.10, 0.10])  # draft model
n = 200_000
x = rng.choice(5, size=n, p=q_draft)
accept = rng.random(n) < np.minimum(1, p[x] / q_draft[x])
leftover = np.maximum(p - q_draft, 0); leftover /= leftover.sum()
out = np.where(accept, x, rng.choice(5, size=n, p=leftover))
empirical = np.bincount(out, minlength=5) / n
print("target p   ", p)
print("empirical  ", empirical.round(3))
print("acceptance rate", round(float(accept.mean()), 3), " overlap sum(min(p,q))", np.minimum(p, q_draft).sum())
alpha, gamma = 0.8, 4
print("tokens per target pass at acceptance 0.8 with 4 drafted:", round((1 - alpha ** (gamma + 1)) / (1 - alpha), 2))
assert np.abs(empirical - p).max() < 0.01

target p    [0.5  0.25 0.15 0.07 0.03]
empirical   [0.499 0.251 0.15  0.07  0.03 ]
acceptance rate 0.752  overlap sum(min(p,q)) 0.75
tokens per target pass at acceptance 0.8 with 4 drafted: 3.36


The empirical distribution matches `p` to within 0.001 on every token, although every proposal came from the draft. 75% of proposals were accepted, which is the overlap of the two distributions. At an acceptance rate of 0.8 with four drafted tokens, one pass of the target yields 3.36 tokens on average. Whether that saves time depends on the cost of the draft model and on the hardware, and this cell measures neither.

## 7. Your turn

Replace each `raise` with your code, then run the test cell under it. Do not edit the tests.

**A. KV-cache size.** Section 2 wrote the product out by hand. Write it as a function you can call for any model in your project.

In [ ]:
# YOUR TURN
def kv_cache_bytes(B, T, L, H_kv, d_h, bytes_per=2):
    """Bytes held by keys and values for batch B, T tokens, L layers, H_kv KV heads of dimension d_h."""
    raise NotImplementedError("your turn")

In [9]:
assert kv_cache_bytes(1, 4096, 24, 8, 64) == 192 * 2**20
assert kv_cache_bytes(2, 8192, 32, 4, 128) == 2**30
assert kv_cache_bytes(1, 4096, 32, 8, 128) == 2**29                       # Llama-3.1-8B at 4,096 tokens: 0.5 GiB
assert kv_cache_bytes(1, 4096, 32, 8, 128, bytes_per=4) == 2**30
print("Llama-3.1-8B, batch 8, 4,096 tokens:", kv_cache_bytes(8, 4096, 32, 8, 128) / 2**30, "GiB")
print("A passed")

Llama-3.1-8B, batch 8, 4,096 tokens: 4.0 GiB
A passed


**B. Per-channel quantisation with rows as channels.** Section 3 treated each column as a channel. PyTorch stores a linear layer's weight as (outputs, inputs), so there each **row** is an output channel. Write `quantise_rows(W, bits)` returning `(q, scale)`, where `q` holds integers in [−qmax, qmax] with qmax = 2^(bits−1) − 1 and `scale` has shape `(rows, 1)`. A row of zeros must get scale 1 so that nothing divides by zero. Then write `dequantise(q, scale)`.

In [ ]:
# YOUR TURN
def quantise_rows(W, bits):
    """Symmetric quantisation with one scale per row. Returns (integer array q, scale of shape (rows, 1))."""
    raise NotImplementedError("your turn")

def dequantise(q, scale):
    """Reconstruct the weights from integers and scales."""
    raise NotImplementedError("your turn")

In [11]:
V = rng.normal(size=(6, 40)); V[0] *= 30; V[3] = 0                          # one large row, one all-zero row
qv, sv = quantise_rows(V, 4)
assert sv.shape == (6, 1) and np.abs(qv).max() <= 7 and np.allclose(qv, np.rint(qv))
assert np.isfinite(dequantise(qv, sv)).all() and np.all(dequantise(qv, sv)[3] == 0)
assert np.all(np.abs(V - dequantise(qv, sv)) <= sv / 2 + 1e-12)               # error is at most half a step in every row
assert np.all(np.abs(qv).max(1)[[0, 1, 2, 4, 5]] == 7)                       # each non-zero row uses the full range
q8, s8 = quantise_rows(V, 8)
print("mean squared error at 4 bits:", round(float(np.mean((V - dequantise(qv, sv)) ** 2)), 5), " at 8 bits:", round(float(np.mean((V - dequantise(q8, s8)) ** 2)), 7))
print("B passed")

mean squared error at 4 bits: 1.11719  at 8 bits: 0.0037734
B passed


## Discussion

1. Why does four-bit weight storage not make all inference memory four-bit?
2. Why can a throughput improvement worsen the latency a user sees?
3. You measured a low weight error after quantisation. What else must you measure before using the model?
4. Your Task 7 vector was found on a 4-bit model. Describe the check you run before submitting it.

## From the toy to your task

Write the memory line of your experiment plan: the model, its parameter count, the precision, the weight memory, the longest context you will run and its KV cache. Task 7 teams add the transfer check from section 5 on a model small enough to load at both precisions. Any timing you report needs the device, batch size, prompt length, output length and a warm-up run beside it.

Submit this notebook with both tests passing and a 150–250-word note: the largest model and precision your hardware can hold, the arithmetic behind that, and one thing the arithmetic leaves out.